# Cooked or Not — train → ensemble → submission (self-contained)

Before Run All, check 3 things:
1. Right panel → Session options → Accelerator = **GPU** (T4/P100).
2. Internet = **ON** (one-time ImageNet weight download).
3. Input tab shows the competition data (auto-attached when created from the competition page).

For a 3–4h run prefer **Save Version → Save & Run All** so it trains in the background.
Output: `submission.csv` (ResNet50 + ResNet18 fold-ensembles, TTA, averaged).

In [ ]:
import os, glob
os.makedirs('src', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
os.environ['PYTHONPATH'] = os.path.abspath('src') + ':' + os.environ.get('PYTHONPATH', '')

def find_data():
    hits = glob.glob('/kaggle/input/**/train_labels.csv', recursive=True)
    if not hits:
        raise FileNotFoundError('competition data not attached (Input tab)')
    return os.path.dirname(hits[0])

DATA = find_data()
print('DATA =', DATA)
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
%%writefile src/dataset.py
"""Clip dataset: 30-frame dashcam folders -> sampled frame tensors."""
import os
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def train_transforms(size=224):
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25),
    ])


def eval_transforms(size=224, hflip=False):
    ops = [transforms.Resize((256, 256)), transforms.CenterCrop(size)]
    if hflip:
        ops.append(transforms.RandomHorizontalFlip(p=1.0))
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)


def sample_indices(n_total=30, n_pick=8, jitter=True, rng=None):
    rng = np.random if rng is None else rng
    base = np.linspace(0, n_total - 1, n_pick)
    if jitter:
        base = base + rng.uniform(-1.0, 1.0, size=n_pick)
    return np.clip(np.round(base), 0, n_total - 1).astype(int)


class ClipDataset(Dataset):
    """split='train' needs train_labels.csv; split='test' reads ids from sample csv."""

    def __init__(self, data_root, split='train', n_frames=8, transform=None,
                 jitter=True, ids=None, seed=0):
        self.clip_dir = os.path.join(data_root, split)
        self.n_frames = n_frames
        self.transform = transform
        self.jitter = jitter
        self.rng = np.random.RandomState(seed)
        if ids is not None:
            self.df = pd.DataFrame({'clip_id': list(ids)})
        elif split == 'train':
            self.df = pd.read_csv(os.path.join(data_root, 'train_labels.csv'))
        else:
            self.df = pd.read_csv(os.path.join(data_root, 'sample_submission.csv'))[['clip_id']]
        self.has_label = 'label' in self.df.columns

    def __len__(self):
        return len(self.df)

    def _load_frame(self, clip_id, idx):
        p = os.path.join(self.clip_dir, clip_id, f'frame_{idx:03d}.jpg')
        return Image.open(p).convert('RGB')

    def __getitem__(self, i):
        row = self.df.iloc[i]
        clip_id = row['clip_id']
        idxs = sample_indices(30, self.n_frames, self.jitter, self.rng)
        frames = [self.transform(self._load_frame(clip_id, j)) for j in idxs]
        import torch
        x = torch.stack(frames)  # (T, C, H, W)
        if self.has_label:
            return x, torch.tensor(float(row['label'])), clip_id
        return x, clip_id


In [ ]:
%%writefile src/model.py
"""Frozen ResNet backbone + attention-MIL temporal head."""
import torch
import torch.nn as nn
from torchvision import models


def build_backbone(name='resnet50', pretrained=True):
    if pretrained:
        try:
            w = {'resnet50': models.ResNet50_Weights.IMAGENET1K_V2,
                 'resnet18': models.ResNet18_Weights.IMAGENET1K_V1}[name]
        except Exception:
            return _legacy_backbone(name)
    else:
        w = None
    base = {'resnet50': models.resnet50, 'resnet18': models.resnet18}[name](weights=w)
    mods = list(base.children())[:-2]  # drop avgpool + fc
    feat = nn.Sequential(*mods, nn.AdaptiveAvgPool2d(1), nn.Flatten())
    dim = {'resnet50': 2048, 'resnet18': 512}[name]
    for p in feat.parameters():
        p.requires_grad = False
    feat.eval()
    return feat, dim


def _legacy_backbone(name):
    base = {'resnet50': models.resnet50, 'resnet18': models.resnet18}[name](pretrained=True)
    mods = list(base.children())[:-2]
    feat = nn.Sequential(*mods, nn.AdaptiveAvgPool2d(1), nn.Flatten())
    dim = {'resnet50': 2048, 'resnet18': 512}[name]
    for p in feat.parameters():
        p.requires_grad = False
    feat.eval()
    return feat, dim


class CrashModel(nn.Module):
    def __init__(self, feat_dim, hidden=512, dropout=0.3):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, hidden), nn.ReLU(),
                                nn.Dropout(dropout))
        self.attn = nn.Linear(hidden, 1)
        self.out = nn.Linear(hidden, 1)

    def forward(self, feats):  # feats: (B, T, D)
        h = self.fc(feats)
        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h * w).sum(dim=1)
        return self.out(ctx).squeeze(1), w.squeeze(2)


class VideoModel(nn.Module):
    """End-to-end wrapper: frames -> frozen backbone -> head."""

    def __init__(self, backbone_name='resnet50', pretrained=True,
                 hidden=512, dropout=0.3):
        super().__init__()
        self.backbone, dim = build_backbone(backbone_name, pretrained)
        self.head = CrashModel(dim, hidden, dropout)

    def forward(self, x):  # x: (B, T, C, H, W)
        B, T = x.shape[:2]
        with torch.no_grad():
            f = self.backbone(x.view(B * T, *x.shape[2:])).view(B, T, -1)
        return self.head(f)


In [ ]:
%%writefile src/train.py
"""GroupKFold training."""
import argparse, os, time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from dataset import ClipDataset, train_transforms, eval_transforms
from model import VideoModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--out', default='models')
    p.add_argument('--backbone', default='resnet50', choices=['resnet50', 'resnet18'])
    p.add_argument('--frames', type=int, default=8)
    p.add_argument('--batch-size', type=int, default=16)
    p.add_argument('--epochs', type=int, default=25)
    p.add_argument('--folds', type=int, default=5)
    p.add_argument('--lr', type=float, default=3e-4)
    p.add_argument('--patience', type=int, default=6)
    p.add_argument('--label-smoothing', type=float, default=0.05)
    p.add_argument('--num-workers', type=int, default=2)
    p.add_argument('--seed', type=int, default=0)
    return p.parse_args()


def run_epoch(model, loader, opt, device, smoothing, train):
    model.train(train)
    crit = torch.nn.BCEWithLogitsLoss()
    tot, n = 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device), y.float().to(device)
        if train:
            opt.zero_grad()
        logits, _ = model(x)
        ys = y * (1 - smoothing) + 0.5 * smoothing
        loss = crit(logits, ys)
        if train:
            loss.backward()
            opt.step()
        tot += loss.item() * len(x)
        n += len(x)
    return tot / max(n, 1)


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    ps, ys, ids = [], [], []
    for x, y, cid in loader:
        logits, _ = model(x.to(device))
        ps += torch.sigmoid(logits).cpu().tolist()
        ys += y.tolist()
        ids += list(cid)
    return np.array(ps), np.array(ys), ids


def main():
    a = parse()
    torch.manual_seed(a.seed)
    np.random.seed(a.seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('device:', device)
    os.makedirs(a.out, exist_ok=True)

    labels = pd.read_csv(os.path.join(a.data, 'train_labels.csv'))
    groups = labels['group_id'].values

    oof = pd.DataFrame({'clip_id': labels['clip_id'], 'label': labels['label'],
                        'pred': np.nan})
    gkf = GroupKFold(n_splits=min(a.folds, len(np.unique(groups))))
    aucs = []
    for fold, (tr, va) in enumerate(gkf.split(labels, labels['label'], groups)):
        tr_ds = ClipDataset(a.data, 'train', a.frames, train_transforms(), True,
                            ids=labels.iloc[tr]['clip_id'], seed=a.seed + fold)
        tr_ds.df = labels.iloc[tr].reset_index(drop=True)
        tr_ds.has_label = True
        va_ds = ClipDataset(a.data, 'train', a.frames, eval_transforms(), False,
                            ids=labels.iloc[va]['clip_id'], seed=999)
        va_ds.df = labels.iloc[va].reset_index(drop=True)
        va_ds.has_label = True
        tr_ld = DataLoader(tr_ds, batch_size=a.batch_size, shuffle=True,
                           num_workers=a.num_workers)
        va_ld = DataLoader(va_ds, batch_size=a.batch_size * 2,
                           num_workers=a.num_workers)

        model = VideoModel(a.backbone, True).to(device)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=a.lr, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, a.epochs)

        best, bad, best_state = 0.0, 0, None
        for ep in range(a.epochs):
            t0 = time.time()
            tl = run_epoch(model, tr_ld, opt, device, a.label_smoothing, True)
            sched.step()
            pv, yv, _ = predict(model, va_ld, device)
            auc = roc_auc_score(yv, pv)
            print(f'fold {fold} ep {ep}: loss {tl:.4f} val_auc {auc:.4f} ({time.time()-t0:.0f}s)', flush=True)
            if auc > best:
                best, bad = auc, 0
                best_state = {k: v.cpu() for k, v in model.head.state_dict().items()}
            else:
                bad += 1
                if bad >= a.patience:
                    break
        model.head.load_state_dict(best_state)
        torch.save({'head': best_state, 'backbone': a.backbone, 'frames': a.frames,
                    'fold': fold, 'auc': best}, f'{a.out}/fold{fold}.pt')
        pv, yv, vids = predict(model, va_ld, device)
        oof.loc[oof.clip_id.isin(vids), 'pred'] = pv
        aucs.append(best)
        print(f'fold {fold} best_auc {best:.4f}')

    oof.to_csv(f'{a.out}/oof.csv', index=False)
    print(f"mean_fold_auc {[round(x, 4) for x in aucs]} -> {np.mean(aucs):.4f}")
    print(f'pooled_oof_auc {roc_auc_score(oof.label, oof.pred):.4f}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/infer.py
"""TTA inference + fold ensemble."""
import argparse, glob, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from dataset import ClipDataset, eval_transforms
from model import VideoModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--models', default='models')
    p.add_argument('--out', default='submission.csv')
    p.add_argument('--crops', type=int, default=4)
    p.add_argument('--batch-size', type=int, default=32)
    p.add_argument('--num-workers', type=int, default=2)
    return p.parse_args()


@torch.no_grad()
def main():
    a = parse()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpts = sorted(glob.glob(f'{a.models}/fold*.pt'))
    assert ckpts, f'no checkpoints in {a.models}/'
    sample = pd.read_csv(os.path.join(a.data, 'sample_submission.csv'))
    ids = sample['clip_id'].tolist()

    acc = np.zeros(len(ids))
    n = 0
    for ck in ckpts:
        c = torch.load(ck, map_location='cpu', weights_only=True)
        for crop in range(a.crops):
            for flip in (False, True):
                ds = ClipDataset(a.data, 'test', c['frames'], eval_transforms(hflip=flip),
                                 True, ids=ids, seed=1000 + crop)
                ld = DataLoader(ds, batch_size=a.batch_size, num_workers=a.num_workers)
                m = VideoModel(c['backbone'], pretrained=False).to(device)
                m.head.load_state_dict(c['head'])
                m.eval()
                ps = []
                for x, _ in ld:
                    logits, _ = m(x.to(device))
                    ps += torch.sigmoid(logits).cpu().tolist()
                acc += np.array(ps)
                n += 1
    sample['label'] = (acc / n).clip(1e-6, 1 - 1e-6)
    sample[['clip_id', 'label']].to_csv(a.out, index=False)
    print(f'wrote {a.out} ({len(sample)} rows, {len(ckpts)} folds x {a.crops} crops x 2 flips)')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile scripts/check_submission.py
"""Reject malformed submissions before upload."""
import argparse, sys
import pandas as pd


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--submission', required=True)
    p.add_argument('--sample', required=True)
    a = p.parse_args()
    sub = pd.read_csv(a.submission)
    smp = pd.read_csv(a.sample)
    assert list(sub.columns) == ['clip_id', 'label'], f'bad columns {list(sub.columns)}'
    assert len(sub) == len(smp), f'{len(sub)} rows, expected {len(smp)}'
    assert sub.clip_id.is_unique, 'duplicate clip_id'
    assert set(sub.clip_id) == set(smp.clip_id), 'clip_id mismatch vs sample'
    assert sub.label.between(0, 1).all(), 'labels outside [0,1]'
    assert sub.label.notna().all(), 'empty values'
    print('submission OK:', len(sub), 'rows')


if __name__ == '__main__':
    try:
        main()
    except AssertionError as e:
        print('INVALID:', e)
        sys.exit(1)


In [ ]:
# Backbone A: ResNet50. ~2h on T4/P100.
!python3 src/train.py --data "$DATA" --out models_r50 --backbone resnet50 --epochs 25 --num-workers 2

In [ ]:
!python3 src/infer.py --data "$DATA" --models models_r50 --out submission_r50.csv --num-workers 2

In [ ]:
# Backbone B: ResNet18 ensemble diversity. ~1h. Skip if GPU quota is tight — R50 alone is ~0.87.
!python3 src/train.py --data "$DATA" --out models_r18 --backbone resnet18 --epochs 25 --num-workers 2

In [ ]:
!python3 src/infer.py --data "$DATA" --models models_r18 --out submission_r18.csv --num-workers 2

In [ ]:
import pandas as pd

a = pd.read_csv('submission_r50.csv')
b = pd.read_csv('submission_r18.csv')
assert (a.clip_id == b.clip_id).all()
sub = a.copy()
sub['label'] = ((a.label + b.label) / 2).clip(1e-6, 1 - 1e-6)
sub.to_csv('submission.csv', index=False)
print('mean pred:', round(float(sub.label.mean()), 3), '(expect ~0.3-0.4: test is ~1/3 crashes)')
print(sub.label.describe().round(3).to_string())

!python3 scripts/check_submission.py --submission submission.csv --sample "$DATA/sample_submission.csv"